# GovOn-EXAONE LoRA v2 머지 & AWQ 양자화 파이프라인

이 노트북은 LoRA v2 어댑터를 베이스 모델에 머지한 뒤 AWQ 양자화를 수행한다.

## 파이프라인 개요

1. **환경 설정** - 호환 버전 패키지 설치
2. **API 키 설정** - WandB & HuggingFace
3. **LoRA 머지** - 베이스 모델 + LoRA v2 어댑터 → BF16 머지 모델
4. **머지 검증** - 추론 테스트로 머지 정상 확인
5. **AWQ 양자화** - W4A16g128 양자화 적용
6. **양자화 검증** - AWQ 모델 추론 테스트
7. **HuggingFace 업로드** - 머지/AWQ 모델 Hub에 업로드
8. **WandB 로깅** - 전체 결과 기록

## 모델 정보

| 항목 | 값 |
|------|----|
| 베이스 모델 | `LGAI-EXAONE/EXAONE-Deep-7.8B` |
| LoRA 어댑터 | `umyunsang/GovOn-EXAONE-LoRA-v2` |
| 머지 출력 | `umyunsang/GovOn-EXAONE-Merged-v2` |
| AWQ 출력 | `umyunsang/GovOn-EXAONE-AWQ-v2` |
| 양자화 설정 | W4A16, group_size=128, zero_point=True |

## 요구사항

- Google Colab GPU 런타임 (T4/L4/A100 권장)
- HuggingFace 토큰 (write 권한)
- WandB API 키

---
## 1. API 키 설정

WandB와 HuggingFace API 키를 입력한다. 이 셀을 먼저 실행하여 키를 설정해야 한다.

In [1]:
# ============================================================
# API 키 설정 (이 셀을 먼저 실행)
# ============================================================
import os
from getpass import getpass

# WandB API 키
WANDB_API_KEY = getpass("WandB API Key: ")
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

# HuggingFace 토큰 (write 권한 필요)
HF_TOKEN = getpass("HuggingFace Token (write 권한): ")
os.environ["HF_TOKEN"] = HF_TOKEN

print("API 키 설정 완료")

WandB API Key: ··········
HuggingFace Token (write 권한): ··········
API 키 설정 완료


---
## 2. 환경 설정

EXAONE 호환 버전으로 패키지를 설치한다.

**중요**: LoRA 학습 시점의 transformers 버전(4.44~4.49)과 EXAONE 모델 코드 revision을 맞춰야 한다.  
최신 transformers(v5+)에서는 `modeling_exaone.py`의 forward pass 구조가 변경되어 LoRA 가중치가 올바르게 적용되지 않는다.

In [2]:
# ============================================================
# 패키지 설치 (EXAONE 호환 버전 고정)
# ============================================================
# 기존 버전 충돌 방지를 위해 제거 후 재설치
!pip uninstall -y transformers accelerate autoawq -q

# 학습 시점 호환 버전 설치
!pip install -q \
    "transformers>=4.44,<4.50" \
    "accelerate>=1.3.0,<2.0" \
    "peft>=0.18.0" \
    "bitsandbytes" \
    "torch>=2.1.0"

# AWQ 양자화 라이브러리
# NOTE: AutoAWQ는 공식적으로 deprecated 상태이나 아직 동작함.
#       향후 vLLM llm-compressor로 전환 예정.
#       버전을 고정하여 향후 breakage 방지.
!pip install -q "autoawq==0.2.7.post3"

# 유틸리티 (protobuf 버전 핀 제거 - Colab 기본 버전과 충돌 방지)
!pip install -q \
    wandb \
    huggingface_hub

print("\n" + "=" * 60)
print("설치 완료. 버전 확인:")
print("=" * 60)

import transformers, peft, accelerate, torch
print(f"  transformers: {transformers.__version__}")
print(f"  peft:         {peft.__version__}")
print(f"  accelerate:   {accelerate.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  CUDA:         {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

# autoawq import 시도 - Colab에서 C 확장 패키지 설치 후 커널 재시작 필요할 수 있음
_awq_import_ok = False
try:
    import awq
    print(f"  autoawq:      {awq.__version__}")
    _awq_import_ok = True
except ImportError:
    print("  autoawq:      import 실패 - 커널을 자동 재시작합니다...")
    print("  ※ 재시작 후 이 셀부터 다시 실행하세요.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

# transformers 버전 검증
tv = transformers.__version__
major, minor = int(tv.split('.')[0]), int(tv.split('.')[1])
if not (major == 4 and 44 <= minor <= 49):
    print(f"\n⚠️  경고: transformers {tv}는 호환 범위(4.44~4.49)를 벗어남!")
    print("    LoRA 어댑터 로드 시 문제가 발생할 수 있다.")
else:
    print(f"\n✅ transformers {tv} - 호환 범위 내")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 4.7 MB/s eta 0:00:00

설치 완료. 버전 확인:
  transformers: 4.49.0
  peft:         0.18.1
  accelerate:   1.13.0
  torch:        2.10.0+cu128
  CUDA:         True (NVIDIA A100-SXM4-40GB)
  autoawq:      0.2.7.post3

✅ transformers 4.49.0 - 호환 범위 내


In [3]:
# ============================================================
# GPU 메모리 확인 및 VRAM 경고
# ============================================================
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3

    print(f"GPU: {gpu_name}")
    print(f"총 VRAM: {total_vram:.1f} GB")
    print(f"가용 VRAM: {free_vram:.1f} GB")

    if total_vram < 16:
        print(f"\n⚠️  경고: VRAM이 {total_vram:.1f}GB로 부족합니다!")
        print("    Stage 1 (머지): ~16GB, Stage 2 (AWQ): ~18-20GB 필요")
        print("    T4 (16GB)에서는 OOM이 발생할 수 있습니다.")
        print("    L4 (24GB) 또는 A100 런타임을 권장합니다.")
    elif total_vram < 24:
        print(f"\n⚠️  참고: VRAM {total_vram:.1f}GB - 머지/AWQ 파이프라인 실행 가능하나 여유가 적습니다.")
        print("    각 Stage 사이 메모리 정리가 중요합니다.")
    else:
        print(f"\n✅ VRAM {total_vram:.1f}GB - 충분합니다.")
else:
    print("⚠️  GPU가 감지되지 않습니다! 런타임 유형을 GPU로 변경하세요.")
    print("  수정 > 노트북 설정 > 하드웨어 가속기 → GPU")

GPU: NVIDIA A100-SXM4-40GB
총 VRAM: 39.5 GB
가용 VRAM: 39.5 GB

✅ VRAM 39.5GB - 충분합니다.


---
## 3. 설정 및 임포트

In [4]:
# ============================================================
# 설정 및 임포트
# ============================================================
import os
import sys
import json
import time
import gc
import re
import warnings
import torch
import wandb
from datetime import datetime
from huggingface_hub import HfApi, hf_hub_download

# ── AutoAWQ deprecation 경고 억제 ──
warnings.filterwarnings("ignore", message=".*AutoAWQ.*deprecated.*")
warnings.filterwarnings("ignore", message=".*autoawq.*deprecated.*")

# ── 모델 설정 ──
BASE_MODEL_ID = "LGAI-EXAONE/EXAONE-Deep-7.8B"
ADAPTER_ID = "umyunsang/GovOn-EXAONE-LoRA-v2"
EXAONE_REVISION = "17b70148e344"  # 학습 호환 마지막 revision (2025-03-19)

# ── 출력 경로 ──
MERGED_OUTPUT_DIR = "/content/models/merged_model"
AWQ_OUTPUT_DIR = "/content/models/awq_quantized_model"

# ── HuggingFace Hub 리포 이름 ──
HF_MERGED_REPO = "umyunsang/GovOn-EXAONE-Merged-v2"
HF_AWQ_REPO = "umyunsang/GovOn-EXAONE-AWQ-v2"

# ── AWQ 양자화 설정 ──
AWQ_QUANT_CONFIG = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

# ── 캘리브레이션 설정 ──
CALIB_N_SAMPLES = 512
CALIB_SEED = 42

# ── WandB 설정 ──
WANDB_PROJECT = "exaone-civil-complaint"

print("설정 완료:")
print(f"  베이스 모델:    {BASE_MODEL_ID}")
print(f"  LoRA 어댑터:    {ADAPTER_ID}")
print(f"  EXAONE 리비전:  {EXAONE_REVISION}")
print(f"  머지 출력:      {MERGED_OUTPUT_DIR}")
print(f"  AWQ 출력:       {AWQ_OUTPUT_DIR}")
print(f"  HF 머지 리포:   {HF_MERGED_REPO}")
print(f"  HF AWQ 리포:    {HF_AWQ_REPO}")
print(f"  AWQ 설정:       {AWQ_QUANT_CONFIG}")

설정 완료:
  베이스 모델:    LGAI-EXAONE/EXAONE-Deep-7.8B
  LoRA 어댑터:    umyunsang/GovOn-EXAONE-LoRA-v2
  EXAONE 리비전:  17b70148e344
  머지 출력:      /content/models/merged_model
  AWQ 출력:       /content/models/awq_quantized_model
  HF 머지 리포:   umyunsang/GovOn-EXAONE-Merged-v2
  HF AWQ 리포:    umyunsang/GovOn-EXAONE-AWQ-v2
  AWQ 설정:       {'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}


---
## 4. Stage 1: LoRA 어댑터 머지

베이스 모델(BF16)에 LoRA v2 어댑터를 머지하여 풀 모델을 생성한다.

### 주의사항
- `revision` 파라미터로 학습 시점의 EXAONE 모델 코드를 고정한다
- 머지 후 파라미터 수가 베이스 모델과 동일한지 검증한다
- LoRA 모듈이 완전히 제거되었는지 확인한다

In [5]:
# ============================================================
# Stage 1: LoRA 어댑터 머지
# ============================================================
merge_start_time = time.time()

# WandB 초기화
run = wandb.init(
    project=WANDB_PROJECT,
    name=f"merge-quantize-v2-{datetime.now().strftime('%Y%m%d-%H%M')}",
    tags=["merge", "awq", "lora-v2", "exaone-7.8b", "pipeline"],
    config={
        "base_model": BASE_MODEL_ID,
        "adapter": ADAPTER_ID,
        "exaone_revision": EXAONE_REVISION,
        "merged_output_dir": MERGED_OUTPUT_DIR,
        "awq_output_dir": AWQ_OUTPUT_DIR,
        "awq_quant_config": AWQ_QUANT_CONFIG,
        "calib_samples": CALIB_N_SAMPLES,
        "stage": "merge_and_quantize_v2",
    }
)

print("=" * 60)
print("Stage 1: LoRA v2 어댑터 머지")
print("=" * 60)

# ── Step 1: 어댑터 설정 검증 ──
print("\n[1/6] 어댑터 설정 검증...")
config_path = hf_hub_download(ADAPTER_ID, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

print(f"  베이스 모델:    {adapter_config['base_model_name_or_path']}")
print(f"  PEFT 타입:      {adapter_config['peft_type']}")
print(f"  Rank (r):       {adapter_config['r']}")
print(f"  Alpha:          {adapter_config['lora_alpha']}")
print(f"  Target modules: {adapter_config['target_modules']}")
print(f"  Dropout:        {adapter_config['lora_dropout']}")

assert adapter_config['base_model_name_or_path'] == BASE_MODEL_ID, \
    f"베이스 모델 불일치: {adapter_config['base_model_name_or_path']} != {BASE_MODEL_ID}"
assert adapter_config['peft_type'] == 'LORA', "LORA 타입이 아님"

wandb.log({"adapter_config": adapter_config})
print("  [OK] 어댑터 설정 검증 완료")

# ── Step 2: 토크나이저 로드 ──
print("\n[2/6] 토크나이저 로드...")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    revision=EXAONE_REVISION,
)
print(f"  Vocab size:        {tokenizer.vocab_size}")
print(f"  Model max length:  {tokenizer.model_max_length}")
print(f"  EOS token:         {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# ── Step 3: 베이스 모델 로드 (BF16) ──
print("\n[3/6] 베이스 모델 로드 (BF16)...")
from transformers import AutoModelForCausalLM

mem_before = torch.cuda.memory_allocated() / 1024**3
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    revision=EXAONE_REVISION,
)
mem_after_base = torch.cuda.memory_allocated() / 1024**3

base_param_count = sum(p.numel() for p in base_model.parameters())
print(f"  베이스 모델 파라미터: {base_param_count:,}")
print(f"  GPU 메모리 사용:     {mem_after_base:.2f} GB")

wandb.log({
    "base_model_params": base_param_count,
    "gpu_mem_base_model_gb": mem_after_base,
})

# ── Step 4: LoRA 어댑터 로드 및 머지 ──
print("\n[4/6] LoRA v2 어댑터 로드 및 머지...")
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
mem_after_adapter = torch.cuda.memory_allocated() / 1024**3

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  전체 파라미터 (어댑터 포함): {total_params:,}")
print(f"  학습 가능 파라미터 (어댑터): {trainable_params:,}")
print(f"  학습 가능 비율:              {100 * trainable_params / total_params:.4f}%")
print(f"  GPU 메모리 (어댑터 포함):    {mem_after_adapter:.2f} GB")

print("\n  어댑터 가중치를 베이스 모델에 머지 중...")
model = model.merge_and_unload()
mem_after_merge = torch.cuda.memory_allocated() / 1024**3

merged_param_count = sum(p.numel() for p in model.parameters())
print(f"  머지 후 파라미터: {merged_param_count:,}")
print(f"  GPU 메모리:       {mem_after_merge:.2f} GB")

# 검증: 파라미터 수 일치 확인
assert merged_param_count == base_param_count, \
    f"파라미터 수 불일치: {merged_param_count} != {base_param_count}"
print("  [OK] 파라미터 수 베이스 모델과 일치")

# 검증: LoRA 모듈 잔존 확인
has_lora = any("lora" in name.lower() for name, _ in model.named_parameters())
assert not has_lora, "머지 후에도 LoRA 모듈이 남아있음!"
print("  [OK] LoRA 모듈 완전 제거 확인")

wandb.log({
    "adapter_trainable_params": trainable_params,
    "adapter_trainable_pct": 100 * trainable_params / total_params,
    "merged_param_count": merged_param_count,
    "gpu_mem_after_merge_gb": mem_after_merge,
    "merge_param_count_match": merged_param_count == base_param_count,
})

# ── Step 5: 머지 모델 저장 ──
print(f"\n[5/6] 머지 모델 저장: {MERGED_OUTPUT_DIR}")
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

# ── 커스텀 코드 파일 복사 (EXAONE은 trust_remote_code 모델) ──
# save_pretrained()는 safetensors/config만 저장하고 커스텀 .py 파일은 저장하지 않음.
# Hub에서 pinned revision의 파일을 다운로드하여 로컬에 복사해야
# AutoAWQ/AutoConfig가 로컬에서 커스텀 클래스를 로드할 수 있음.
import shutil
custom_code_files = [
    "configuration_exaone.py",
    "modeling_exaone.py",
]
print("  커스텀 코드 파일 복사 (pinned revision)...")
for fname in custom_code_files:
    try:
        src = hf_hub_download(
            BASE_MODEL_ID,
            fname,
            revision=EXAONE_REVISION,
        )
        dst = os.path.join(MERGED_OUTPUT_DIR, fname)
        shutil.copy2(src, dst)
        print(f"    {fname} → 복사 완료")
    except Exception as e:
        print(f"    {fname} → 복사 실패: {e}")

# ── config.json auto_map 패치: 원격 Hub 참조 → 로컬 파일 참조 ──
# save_pretrained()가 저장한 config.json의 auto_map은 원격 리포를 참조함:
#   "LGAI-EXAONE/EXAONE-Deep-7.8B--configuration_exaone.ExaoneConfig"
# AutoAWQ가 AutoConfig.from_pretrained()를 호출하면 최신 Hub 코드를 다운로드하여
# transformers 4.44~4.49와 호환되지 않는 RopeParameters 등의 import 오류가 발생.
# 로컬 참조로 변경하면 위에서 복사한 pinned revision 파일을 직접 사용함.
config_json_path = os.path.join(MERGED_OUTPUT_DIR, "config.json")
with open(config_json_path, 'r') as f:
    config_data = json.load(f)

if "auto_map" in config_data:
    patched_auto_map = {}
    for key, value in config_data["auto_map"].items():
        # "LGAI-EXAONE/EXAONE-Deep-7.8B--configuration_exaone.ExaoneConfig"
        # → "configuration_exaone.ExaoneConfig"
        if "--" in value:
            value = value.split("--", 1)[1]
        patched_auto_map[key] = value
    config_data["auto_map"] = patched_auto_map
    with open(config_json_path, 'w') as f:
        json.dump(config_data, f, indent=2, ensure_ascii=False)
    print(f"  config.json auto_map 패치 완료: {patched_auto_map}")

# 모델 크기 계산
merged_size_bytes = sum(
    os.path.getsize(os.path.join(MERGED_OUTPUT_DIR, f))
    for f in os.listdir(MERGED_OUTPUT_DIR)
    if f.endswith(('.safetensors', '.bin'))
)
merged_size_gb = merged_size_bytes / 1024**3
print(f"  디스크 크기: {merged_size_gb:.2f} GB")

# ── Step 6: 머지 모델 추론 테스트 ──
print("\n[6/6] 머지 모델 추론 테스트...")

def get_eos_ids(tok):
    """EXAONE EOS 토큰 ID 목록 반환"""
    ids = [tok.eos_token_id]
    for special_tok in ['[|endofturn|]', '<|endofturn|>']:
        tid = tok.convert_tokens_to_ids(special_tok)
        if tid is not None and tid != tok.unk_token_id and tid not in ids:
            ids.append(tid)
    return ids

test_prompts = [
    "주민등록증 재발급 절차가 어떻게 되나요?",
    "우리 동네 도로에 포트홀이 생겨서 위험합니다.",
]

eos_ids = get_eos_ids(tokenizer)
print(f"  EOS 토큰 IDs: {eos_ids}")

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."},
        {"role": "user", "content": f"다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n민원 내용: {prompt}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos_ids,
        )

    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=False)
    clean_response = re.sub(r"<thought>.*?</thought>", "", response, flags=re.DOTALL).strip()

    print(f"\n  --- 테스트 {i+1} ---")
    print(f"  질의: {prompt}")
    print(f"  응답 (300자): {clean_response[:300]}")
    has_eos = any(tok in response for tok in ['[|endofturn|]', '<|endofturn|>'])
    print(f"  EOS 생성: {'Yes' if has_eos else 'No'}")

merge_elapsed = time.time() - merge_start_time

wandb.log({
    "merged_model_size_gb": merged_size_gb,
    "merge_time_seconds": merge_elapsed,
})

# 머지 로그 저장
merge_log = {
    "stage": "1_lora_merge_v2",
    "timestamp": datetime.now().isoformat(),
    "base_model": BASE_MODEL_ID,
    "adapter": ADAPTER_ID,
    "exaone_revision": EXAONE_REVISION,
    "base_param_count": base_param_count,
    "adapter_trainable_params": trainable_params,
    "merged_param_count": merged_param_count,
    "model_size_gb": merged_size_gb,
    "elapsed_seconds": merge_elapsed,
    "adapter_config": adapter_config,
}
with open(os.path.join(MERGED_OUTPUT_DIR, "merge_log.json"), "w") as f:
    json.dump(merge_log, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{'=' * 60}")
print(f"Stage 1 완료! 소요 시간: {merge_elapsed:.1f}초 ({merge_elapsed/60:.1f}분)")
print(f"머지 모델 저장 위치: {MERGED_OUTPUT_DIR}")
print(f"{'=' * 60}")

# 메모리 정리 (AWQ에서 다시 로드하므로 여기서 해제)
del model, base_model
gc.collect()
torch.cuda.empty_cache()
print("\nGPU 메모리 해제 완료")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: umyunsang (umyun3) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Stage 1: LoRA v2 어댑터 머지

[1/6] 어댑터 설정 검증...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

  베이스 모델:    LGAI-EXAONE/EXAONE-Deep-7.8B
  PEFT 타입:      LORA
  Rank (r):       16
  Alpha:          32
  Target modules: ['v_proj', 'k_proj', 'o_proj', 'gate_proj', 'q_proj', 'down_proj', 'up_proj']
  Dropout:        0.05
  [OK] 어댑터 설정 검증 완료

[2/6] 토크나이저 로드...


...

  베이스 모델 파라미터: 7,818,448,896
  GPU 메모리 사용:     14.56 GB

[4/6] LoRA v2 어댑터 로드 및 머지...


adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

  전체 파라미터 (어댑터 포함): 7,827,886,080
  학습 가능 파라미터 (어댑터): 0
  학습 가능 비율:              0.0000%
  GPU 메모리 (어댑터 포함):    14.60 GB

  어댑터 가중치를 베이스 모델에 머지 중...
  머지 후 파라미터: 7,818,448,896
  GPU 메모리:       14.57 GB
  [OK] 파라미터 수 베이스 모델과 일치
  [OK] LoRA 모듈 완전 제거 확인

[5/6] 머지 모델 저장: /content/models/merged_model
  커스텀 코드 파일 복사 (pinned revision)...
    configuration_exaone.py → 복사 완료
    modeling_exaone.py → 복사 완료
  config.json auto_map 패치 완료: {'AutoConfig': 'configuration_exaone.ExaoneConfig', 'AutoModelForCausalLM': 'modeling_exaone.ExaoneForCausalLM', 'AutoModelForSequenceClassification': 'modeling_exaone.ExaoneForSequenceClassification'}
  디스크 크기: 14.56 GB

[6/6] 머지 모델 추론 테스트...
  EOS 토큰 IDs: [361]


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  --- 테스트 1 ---
  질의: 주민등록증 재발급 절차가 어떻게 되나요?
  응답 (300자): 안녕하십니까?
귀하께서 응답소(민원상담)를 통해 신청하신 민원에 대한 검토 결과를 다음과 같이 알려드립니다.
먼저, 귀하의 민원내용은 "주민등록증 재발급"에 관한 것으로 판단됩니다.
귀하의 질의사항에 대해 검토한 의견은 다음과 같습니다.
1. 주민등록증 재발급은 주민등록법 제8조제2항 및 동법 시행령 제9조제3항에 의거하여, 주민등록증 발급신청자 또는 발급받은 사람이 본인이나 가족의 신분증을 분실·손상 등으로 사용할 수 없게 된 경우, 해당 주민센터에서 재발급 신청 가능합니다.
2. 재발급 신청 시, 본인 확인을 위해 신분증을 지참
  EOS 생성: No

  --- 테스트 2 ---
  질의: 우리 동네 도로에 포트홀이 생겨서 위험합니다.
  응답 (300자): 안녕하십니까? 귀하께서 신청하신 민원에 대한 검토 결과를 다음과 같이 알려드립니다.
귀하의 민원내용은 "도로 포트홀 보수"에 관한 것으로 판단됩니다.
귀하의 질의사항에 대해 검토한 의견은 다음과 같습니다.
가. 현재 해당 구간(도로명 : 도로)은 포트홀 발생지역으로, 포트홀 발생 시 즉시 보수하여 안전한 통행이 되도록 조치 중에 있음을 알려드리며,
나. 포트홀 발생 지역은 도로 표면의 노후화로 인한 것으로 추정되어, 포트홀 발생 시 보수 후에도 동일한 위치에서 반복적으로 발생할 수 있으므로, 지속적인 관리 및 점검이 필요함을 알려드
  EOS 생성: Yes

Stage 1 완료! 소요 시간: 138.8초 (2.3분)
머지 모델 저장 위치: /content/models/merged_model

GPU 메모리 해제 완료


---
## 5. Stage 2: AWQ 양자화

머지된 BF16 모델에 AWQ(Activation-aware Weight Quantization)를 적용한다.

- **양자화 방식**: W4A16g128 (4bit 가중치, 16bit 활성화, group_size=128)
- **캘리브레이션**: 민원 도메인 데이터 512개 샘플 사용
- **예상 크기**: BF16 ~15GB → AWQ ~4.5GB (약 3.3x 압축)

In [7]:
# ============================================================
# Stage 2: AWQ 양자화
# ============================================================
import random
import urllib.request

quant_start_time = time.time()

print("=" * 60)
print("Stage 2: AWQ 양자화 (W4A16g128)")
print("=" * 60)

# ── GPU 메모리 정리 (Stage 1에서 남은 잔여 메모리 해제) ──
gc.collect()
torch.cuda.empty_cache()
mem_cleaned = torch.cuda.memory_allocated() / 1024**3
print(f"  GPU 메모리 정리 후: {mem_cleaned:.2f} GB 사용 중")

# ── Step 1: 캘리브레이션 데이터 준비 ──
print("\n[1/4] 캘리브레이션 데이터 준비...")

# 토크나이저 재로드 (머지 모델에서)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MERGED_OUTPUT_DIR, trust_remote_code=True)

def prepare_calibration_data(n_samples=512, seed=42):
    """
    민원 도메인 캘리브레이션 데이터를 준비한다.

    우선순위:
      1) 로컬 v2_train.jsonl 파일
      2) GitHub raw URL에서 다운로드
      3) 인라인 폴백 (56개 다양한 예제, 8개 카테고리 전체 커버)

    Returns:
        List[str]: EXAONE 챗 템플릿 형식의 텍스트 리스트
    """
    random.seed(seed)

    def _load_from_jsonl(data_path, source_label):
        """JSONL 파일에서 캘리브레이션 샘플을 로드한다."""
        print(f"  {source_label}: {data_path}")
        samples = []
        with open(data_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        random.shuffle(lines)

        for line in lines[:n_samples * 2]:
            try:
                item = json.loads(line.strip())
                text = item.get('text', '')
                if len(text) > 100:
                    samples.append(text)
            except (json.JSONDecodeError, KeyError):
                continue
            if len(samples) >= n_samples:
                break

        if len(samples) >= n_samples // 2:
            print(f"  캘리브레이션 샘플 {len(samples)}개 로드 완료")
            return samples
        return None

    # ── 방법 1: 로컬 데이터 파일 ──
    local_data_paths = [
        "/content/ondevice-ai-civil-complaint/data/processed/v2_train.jsonl",
        "/content/data/v2_train.jsonl",
        "data/processed/v2_train.jsonl",
    ]

    for data_path in local_data_paths:
        if os.path.exists(data_path):
            result = _load_from_jsonl(data_path, "로컬 데이터 사용")
            if result:
                return result

    # ── 방법 2: GitHub에서 다운로드 ──
    github_urls = [
        "https://raw.githubusercontent.com/GovOn-org/GovOn/develop/data/processed/v2_train.jsonl",
        "https://raw.githubusercontent.com/GovOn-org/GovOn/main/data/processed/v2_train.jsonl",
    ]

    for url in github_urls:
        try:
            print(f"  GitHub에서 다운로드 시도: {url[:80]}...")
            tmp_path = "/tmp/v2_train_calib.jsonl"
            urllib.request.urlretrieve(url, tmp_path)
            result = _load_from_jsonl(tmp_path, "GitHub 다운로드 데이터 사용")
            if result:
                return result
        except Exception as e:
            print(f"  다운로드 실패: {e}")
            continue

    # ── 방법 3: 인라인 폴백 (56개 다양한 예제, 8개 카테고리 전체 커버) ──
    print("  로컬/원격 데이터 없음 → 인라인 폴백 데이터 사용 (56개 예제, 8개 카테고리)")

    # 실제 v2_train.jsonl과 동일한 형식으로 구성
    # 카테고리별 7개씩 = 56개, 다양한 답변 스타일과 길이 포함
    system_msg = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."

    # (카테고리, 민원내용, 답변) 튜플 리스트
    examples = [
        # ────────── 교통 (7개) ──────────
        ("교통", "택시 신고하려고 전화했어요!",
         "상황 확인하였습니다. 신고접수 해드렸고요. 일주일 이내로 신고결과 연락드리도록 하겠습니다."),

        ("교통", "삼청동에서 종로구가는 버스를 알고싶습니다. 시간은 얼마나 걸립니까? 첫차는 언제입니까? 차비는 얼마입니까?",
         "삼청공원에서 북촌한옥마을로 가는 버스는 470버스를 타시면 됩니다. 20분정도 걸립니다. 오전 6시 25분입니다. 1,150원 입니다."),

        ("교통", "서울에서 강릉역가는 ktx문의합니다 서울역에서만 ktx 탈수 있는건지요? 청량리역에서도 탈수 있다고요?",
         "강릉역까지 운행하는 ktx는 서울역에서 출발하여 청량리역 그리고 망우역을 경유합니다. 네 맞습니다. 평일기준 14번 운행합니다. 9시22분 열차가 있습니다. 소요시간은 1시간 26분입니다. 성인요금 26,000원입니다."),

        ("교통", "도로에 포트홀이 생겨서 위험합니다. 정확한 위치는 OO로 123번길 부근입니다.",
         "도로 포트홀 신고 감사합니다. 해당 위치를 확인하여 긴급 보수 조치하겠습니다. 도로관리과에서 현장 확인 후 48시간 이내 보수 완료 예정입니다. 추가 위험 사항 발견 시 도로관리과(OOO-OOOO)로 직접 연락 부탁드립니다."),

        ("교통", "버스 노선 변경 요청을 하고 싶습니다. 현재 마을버스가 우리 아파트 단지를 경유하지 않습니다.",
         "버스 노선 변경 건의를 접수하겠습니다. 노선 변경은 교통수요 분석, 주민 의견 수렴, 관계기관 협의 등의 절차가 필요하며 통상 3~6개월의 검토 기간이 소요됩니다. 교통정책과에서 해당 노선의 승객 수요를 분석한 후 결과를 안내드리겠습니다."),

        ("교통", "신호등이 고장났습니다. 깜빡이만 계속 들어옵니다.",
         "신호등 고장 신고 감사합니다. 교통안전시설 관리 부서에 즉시 전달하여 점검 및 수리 조치하겠습니다. 해당 교차로의 정확한 위치를 알려주시면 더욱 신속한 처리가 가능합니다. 긴급 시 경찰서 교통과(112)에도 신고 부탁드립니다."),

        ("교통", "KTX 지연 신청 어디에 해요? 시간은 왜 물어보는 거예요? 저는 얼마나 보상받을 수 있어요?",
         "지연된 시간에 따라 차등 지원됩니다. 1시간 이상 지연되었다면 반환 50%와 할인증 100% 중 선택하실 수 있습니다. 사용 기한은 1년 이내입니다. 코레일 앱을 설치하셔서 마이페이지에서 열차운행중지 및 지연배상 신청으로 들어가시면 됩니다."),

        # ────────── 행정 (7개) ──────────
        ("행정", "주민등록증 재발급 절차가 어떻게 되나요?",
         "주민등록증 재발급은 주민센터(읍면동사무소)를 방문하시거나 정부24 온라인을 통해 신청하실 수 있습니다. 방문 시 본인 확인을 위한 신분증(운전면허증 등)이 필요하며, 수수료는 5,000원입니다. 처리 기간은 약 3~5일 정도 소요됩니다."),

        ("행정", "졸업증명서 발급 가능한가요? 비용은요?",
         "네 조회해 드리겠습니다. 정확한 학번 부탁드리겠습니다. 2부 출력할게요. 출력하시면 장 수당 1,200원입니다."),

        ("행정", "전입신고는 어디서 하나요? 기한이 있나요?",
         "전입신고는 새로운 거주지의 주민센터를 방문하시거나 정부24(www.gov.kr) 온라인으로 신청하실 수 있습니다. 전입일로부터 14일 이내에 신고하셔야 합니다. 세대주 또는 세대원이 신고할 수 있으며, 필요 서류는 신분증입니다."),

        ("행정", "인감증명서 발급에 필요한 서류는 무엇인가요?",
         "인감증명서 발급을 위해서는 본인이 직접 주민센터를 방문하셔야 합니다. 본인 확인을 위한 신분증(주민등록증, 운전면허증 등)을 지참하시면 됩니다. 대리인 발급 시에는 위임장과 대리인 신분증이 추가로 필요합니다. 수수료는 600원입니다."),

        ("행정", "이사를 가면 청년수당 못 받게 되는 건가요? 받을 수 있는 방법이 없는건가요?",
         "서울로 주소지가 등록이 되어 있어야만 가능합니다. 이사를 가게 되면 직접 고용센터로 연락을 해 주셔야 합니다. 해당 지역에서 유사한 청년 지원 사업이 있는지 문의해 보시는 것을 권장합니다. 청년수당은 6개월 동안 받을 수 있습니다."),

        ("행정", "문화생활 이용가능한곳 있을까요? 예약은 어디서 하나요?",
         "서울 박물관, 한성백제박물관, 서울역사박물관, 수도박물관 등이 있습니다. 현재 사전예약제로 운영되고 있으며 관람 인원은 1일 5회(회차당 100명 제한)입니다. 서울시공공서비스예약(https://yeyak.seoul.go.kr/)에서 하시면 됩니다. 매주 월요일, 1월 1일에 휴관합니다."),

        ("행정", "디지털 성범죄에 따른 접수가 가능한 온 서울 세이프 맞춤지원에 대해 알고 싶습니다.",
         "온 서울 세이프는 서울시가 전국 최초로 운영하는 디지털 성범죄 피해구제 서비스입니다. 예방부터 피해 발견, 삭제 지원, 수사 지원, 사후 관리까지 전 과정을 피해자 입장에서 지원합니다. 수사기관에서 피해를 진술할 때도 동석이 가능합니다."),

        # ────────── 환경 (7개) ──────────
        ("환경", "수도요금을 못냈는데 고지서가 없어져서 얼마를 내야하는지 모르겠어서 연락드렸습니다.",
         "네 확인해드리겠습니다. 현재 5월 요금 납부가 안되어 있으십니다. 36,030원입니다. 현재 선생님 댁은 격월로 납부하시는데 홀수달마다 납부하시는걸로 신청되어있겠습니다. 매달 납입하는 걸로 변경 등록해드렸습니다."),

        ("환경", "불법 쓰레기 투기를 신고합니다. 매일 밤 같은 장소에 투기가 반복됩니다.",
         "불법 쓰레기 투기 신고 감사합니다. 환경관리과에서 현장 확인 후 수거 조치하겠습니다. CCTV 확인 등을 통해 투기자를 확인할 경우 폐기물관리법에 따라 과태료가 부과됩니다. 반복 투기 지역이므로 단속 강화 조치를 취하겠습니다."),

        ("환경", "소음 민원을 접수하고 싶습니다. 옆집 공사 소음이 심합니다.",
         "소음 민원 접수 감사합니다. 공사장 소음의 경우 환경과에서 소음 측정 및 현장 점검을 실시합니다. 소음·진동관리법에 따라 생활소음 규제기준을 초과할 경우 시정명령이 내려질 수 있습니다. 공사 시간은 평일 오전 8시~오후 6시로 제한됩니다."),

        ("환경", "수도배관 공사를 했는데도 녹물이 나올수 있나요? 옥내배관요?",
         "네. 건물 전체 공용배관은 정상일듯 합니다. 각 세대에 공급되는 옥내배관이 부식되어 녹물이 나올 수 있습니다. 세대별 옥내배관 교체를 하시거나 수도계량기 녹물 필터 설치로 어느정도 문제가 완화될 수 있습니다."),

        ("환경", "수도세가 들쭉날쭉해서요. 혹시 기본요금이 매월 달라지나요?",
         "아니요, 기본요금은 정해져 있습니다. 수도요금은 기본요금, 상수도, 하수도로 구성되어 있고, 사용량이 많을수록 높은 단가가 적용되는 누진요율이 발생합니다. 아리수 사이버고객센터에서 요금계산을 해보실 수 있습니다."),

        ("환경", "물탱크를 청소하고 싶습니다.",
         "안전을 위해 저수조청소업자를 통한 청소를 권장합니다. 에스엠씨 OOO-OOOO-OOOO번으로 문의하시면 됩니다. 저수조 청소는 연 1회 이상 실시하도록 수도법에서 규정하고 있습니다."),

        ("환경", "요금감면제도라는게 있다는데 받을 수 있나요?",
         "요금감면제도는 기초생활수급자 감면, 누수요금 감면, 전자고지 자동납부 감면, 자가검침 감면, 세대분할 감면, 다자녀 하수도사용료 감면, 독립유공자 감면이 있습니다. 기초생활수급자는 월 사용량 10톤을 감면받으며, 관할 주민센터 또는 수도사업소에 방문하여 신청하시기 바랍니다."),

        # ────────── 세금 (7개) ──────────
        ("세금", "재산세 납부 기한이 언제인가요?",
         "재산세는 매년 7월과 9월에 나누어 납부합니다. 7월에는 건축물분과 주택분(1/2), 9월에는 토지분과 주택분(나머지 1/2)이 부과됩니다. 납부 기한은 각각 7월 31일, 9월 30일까지입니다."),

        ("세금", "자동차세 연납 신청을 하고 싶습니다.",
         "자동차세 연납 신청은 위택스(www.wetax.go.kr) 또는 관할 구청 세무과에서 가능합니다. 1월에 연납 신청 시 약 9.15%의 세액 공제 혜택을 받으실 수 있습니다. 3월, 6월, 9월에도 신청 가능하나 공제율이 다릅니다."),

        ("세금", "근로장려금은 어떻게 결정되나요?",
         "조세특례제한법 제100조의7 제2항에 따르면, 납세지 관할 세무서장은 신청을 받은 경우 제100조의5에 따라 산정한 금액의 100분의 95에 해당하는 금액을 근로장려금으로 결정합니다."),

        ("세금", "지방세 환급금 신청은 어떻게 해야하죠? 어느 사이트에서 신청을 하나요?",
         "인터넷에서 접수를 하셔야 합니다. ETAX에 접속하여 환급금 조회하시고 신청을 하셔야 합니다. 신한은행에서 현금수령 가능하며, 전국 전지점에서 수령이 가능합니다. 50만원 이상일 경우 본인 계좌로만 지급합니다. 전화 또는 팩스, 우편으로도 신청 가능합니다."),

        ("세금", "1989.8.1. 개정 전 소득세법 시행령 및 시행규칙의 효력은 유효한가요?",
         "1989.8.1. 대통령령 제12767호로 개정되기 전의 소득세법 시행령 제170조 제1항 단서 및 제115조 제1항 제1호 (다)목과 같은법 시행규칙 제56조의5 제5항은 유효한 규정입니다. 이 규정들은 헌법 제38조와 제59조에 따라 유효하게 적용된 것으로 판단됩니다."),

        ("세금", "직권조사 사항인 제소 요건의 구비 여부는 당사자의 진술에 구애받지 않아도 되나요?",
         "제소 요건의 구비 여부는 직권조사 사항으로서 반드시 당사자의 진술에 구애될 필요는 없습니다. 원심은 원고가 기간연장통지를 받은 사실이 없다는 진술에만 의거하여 제소기간이 도과되었다고 판단하였으나, 이러한 제소요건의 구비 여부는 직권조사 사항으로 당사자의 진술에만 의존하는 것은 위법하다고 판시하였습니다."),

        ("세금", "구 소득세법 제108조 제1항이 과세요건 명확주의에 위반됩니까?",
         "구 소득세법 제108조 제1항은 예정신고와 함께 자진신고를 이행한 경우에만 세액공제를 인정하고 있습니다. 해당 조항의 문언 자체를 살펴보면, 납세의무자가 자기 명의로 예정신고 납부를 모두 이행한 경우에만 세액공제 혜택을 부여하는 것이 명확합니다. 따라서 과세요건 명확주의에 위반된다고 볼 수 없습니다."),

        # ────────── 기타 (7개) ──────────
        ("기타", "혈압약 및 고지혈증약 복용중인 70세 화성시 읍 주민입니다. 가까운 건강 증진 센터를 두고 멀리 보건소로 가야 하는 불편을 해소해주세요.",
         "안녕하십니까? 귀하께서 신청하신 민원에 대한 검토 결과를 다음과 같이 알려드립니다. 귀하의 민원 내용은 주민 건강 지원센터 일차진료 및 처방 개시 요청에 관한 것으로 판단됩니다. 현재 코로나19 상황으로 중단된 일차진료 서비스의 재개를 관련 부서에 전달하겠습니다."),

        ("기타", "경비실에서 분리수거 제대로 안 됐다고 연락이 왔는데 면 종류는 일반 쓰레기인데도 다른 일반 쓰레기랑 또 따로 분리해서 버려야 하나요?",
         "일반 면 소재(수건, 부직포 가방 등)는 종량제 봉투에 담아 일반 쓰레기로 배출하시면 됩니다. 관련 법규상 면 종류를 별도로 분리하여 배출해야 한다는 규정은 없습니다. 다만 의류의 경우 의류수거함에 배출 가능합니다."),

        ("기타", "강정해오름노을길 분수대 멋진데 언제부터 언제까지 운영하나요?",
         "안녕하십니까, 제주특별자치도입니다. 해오름노을길 입구에 위치한 바닥분수는 8월 11일 운영 예정으로 9월 15일까지 평일(공휴일 포함) 1일 5회(11시, 13시, 14시, 15시, 16시), 회당 30분 가동할 예정입니다."),

        ("기타", "조릿대가 한라산을 거의 다 덮고 있어서 안타깝습니다. 제거 속도보다 확산 속도가 더 빠른 건 아닌지 궁금합니다.",
         "한라산 조릿대 문제에 관심을 가져주셔서 감사합니다. 현재 제주도에서는 연간 일정 면적의 조릿대 제거 사업을 진행하고 있으며, 자생식물 복원을 병행하고 있습니다. 구체적인 제거 계획과 진행 현황은 한라산국립공원관리소에 문의하시면 상세한 안내를 받으실 수 있습니다."),

        ("기타", "공원 벤치가 파손되어 있어서 위험합니다. 수리 요청드립니다.",
         "공원 시설물 파손 신고 감사합니다. 공원녹지과에서 현장 확인 후 수리 조치하겠습니다. 이용자 안전을 위해 파손 벤치에 안전 표시를 우선 조치하고 7일 이내 수리를 완료하도록 하겠습니다."),

        ("기타", "주말 농장을 운영하고 싶은데 절차가 어떻게 되나요?",
         "주말농장 운영을 위해서는 농업경영체 등록과 함께 해당 농지의 용도 확인이 필요합니다. 농지법에 따라 주말체험영농 목적의 농지 임대차는 1,000제곱미터 미만으로 제한됩니다. 상세한 절차는 시·군 농업기술센터에 문의하시기 바랍니다."),

        ("기타", "반려동물 등록은 어디서 하나요? 비용은 얼마인가요?",
         "반려동물 등록은 동물병원 또는 동물보호관리시스템(www.animal.go.kr)을 통해 가능합니다. 등록 대상은 2개월 이상의 개이며, 내장형 무선식별장치(마이크로칩) 삽입 시 동물병원별로 비용이 상이합니다. 동물등록을 하지 않을 경우 과태료가 부과될 수 있습니다."),

        # ────────── 건축 (7개) ──────────
        ("건축", "산업단지 예정부지 면적이 15만 제곱미터 미만인 경우, 민간 사업시행자가 전략환경영향평가서를 작성할 수 있나요?",
         "산업단지 예정부지의 면적이 15만 제곱미터 미만인 경우, 민간 사업시행자는 산업단지 인·허가 절차 간소화를 위한 특례법 제23조제1항에 따라 전략환경영향평가서를 작성할 수 있습니다. 이 법은 환경영향평가법의 규정에 대한 특례를 두고 있습니다."),

        ("건축", "수선이 수반되지 않는 용도변경 허가 시에도 건축물 외벽의 창호를 방화유리창으로 해야 하나요?",
         "2021년 7월 5일 이후 건축물 외벽에 설치하는 창호의 수선 등이 수반되지 않는 용도변경 허가를 신청하거나 신고하는 경우에도, 건축법 제52조제4항에 따라 건축물 외벽에 설치된 창호를 방화유리창으로 해야 합니다."),

        ("건축", "노후 연면적 기준도 주택 정비형 재개발사업 정비계획 입안 요건에 해당하는지요?",
         "안녕하십니까? 민원상담을 통해 접수하신 민원사항에 대해 답변드립니다. 노후 연면적(2/3) 기준은 주택정비형 재개발사업 정비계획 입안 요건에 해당합니다. 정비구역 지정을 위한 주민 입안 동의율 변경 사항은 관련 조례 개정 여부를 확인하시기 바랍니다."),

        ("건축", "건축허가를 받기 위한 절차가 어떻게 되나요?",
         "건축허가 절차는 다음과 같습니다. 먼저 건축설계 도서를 작성하시고, 관할 구청 건축과에 허가 신청서를 제출합니다. 건축법 제11조에 따라 심의가 필요한 경우 건축위원회 심의를 거치며, 통상 처리 기간은 15~30일입니다."),

        ("건축", "설계도서의 작성에서 대통령령으로 정하는 건축물은 무엇인가요?",
         "건축법 시행령 제18조에 따르면, 읍·면지역에서 건축하는 건축물 중 연면적이 200제곱미터 이하인 창고 및 농막과 연면적 400제곱미터 이하인 축사, 작물재배사 등이 이에 해당합니다."),

        ("건축", "불법건축물 신고는 어떻게 하나요?",
         "불법건축물 신고는 관할 구청 건축과 또는 국민신문고(www.epeople.go.kr)를 통해 가능합니다. 신고 접수 후 현장 조사를 실시하며, 건축법 위반이 확인될 경우 이행강제금 부과 및 시정명령이 내려집니다."),

        ("건축", "리모델링과 재건축의 차이가 무엇인가요?",
         "리모델링은 기존 건축물의 대수선 또는 증축을 통해 건물의 기능을 개선하는 것이고, 재건축은 노후 건축물을 철거하고 새로 건설하는 것입니다. 리모델링은 건축법에 따라 처리되고, 재건축은 도시 및 주거환경정비법에 따라 진행됩니다."),

        # ────────── 복지 (7개) ──────────
        ("복지", "기초생활수급자 신청 방법을 알려주세요.",
         "기초생활수급자 신청은 주소지 주민센터(읍면동사무소)를 방문하여 신청하실 수 있습니다. 신청 시 소득·재산 조사를 거치며, 가구의 소득인정액이 기준 중위소득의 일정 비율 이하인 경우 수급자로 선정됩니다. 필요 서류는 신분증, 소득·재산 증빙서류 등입니다."),

        ("복지", "장애인 등록 절차가 궁금합니다.",
         "장애인 등록은 주소지 주민센터(읍면동사무소)에서 신청하실 수 있습니다. 신청 후 국민연금공단에서 장애 정도를 심사하며, 심사 결과에 따라 장애인으로 등록됩니다. 장애 정도는 심한 장애와 심하지 않은 장애로 구분됩니다."),

        ("복지", "고객의 폭언 등으로 인해 근로자에게 건강장해가 발생할 우려가 있는 경우, 사업주의 조치 의무는 무엇인가요?",
         "산업안전보건법 제41조제2항에 따라 사업주는 고객응대근로자의 건강장해 예방을 위해 필요한 조치를 해야 합니다. 구체적으로는 업무의 일시적 중단, 전환, 심리상담 지원 등이 포함됩니다. 다만 이를 위해 개인정보를 목적 외로 사용하거나 제3자에게 제공하는 것은 허용되지 않습니다."),

        ("복지", "장애학생지원센터가 시각장애인 등의 교육을 위해 설치된 시설에 해당하나요?",
         "장애학생지원센터는 장애인 등에 대한 특수교육법 제30조에 따라 설치되며, 시각장애인을 포함한 장애학생의 교육과 생활을 지원하는 비영리 목적의 시설입니다. 따라서 저작권법 시행령 제14조 제1항 제3호에 따른 시설에 해당합니다."),

        ("복지", "공중보건의사가 근무기간 연장 명령을 받으면 채용계약은 어떻게 되나요?",
         "농어촌 등 보건의료를 위한 특별조치법 제9조 제6항에 따르면, 공중보건의사가 보건복지부장관의 근무기간 연장 명령을 받은 경우, 채용계약 기간이 연장된 것으로 봅니다."),

        ("복지", "위원회의 위원의 임기는 어떻게 정해지나요?",
         "사회보장기본법 제21조 제4항에 따르면 위원의 임기는 2년으로 하며, 공무원인 위원의 임기는 그 재임 기간으로 하고, 기관·단체의 대표자 자격으로 위촉된 경우에는 그 대표 지위를 유지하는 기간으로 합니다."),

        ("복지", "소상공인 지원금 신청 방법을 알려주세요.",
         "소상공인 지원금은 소상공인시장진흥공단 또는 지자체 경제진흥과를 통해 신청하실 수 있습니다. 지원 대상, 신청 기간, 필요 서류 등은 사업별로 상이하므로 해당 기관에 사전 문의하시기 바랍니다. 온라인 신청은 소상공인시장진흥공단 홈페이지에서 가능합니다."),

        # ────────── 안전 (7개) ──────────
        ("안전", "승강기 안전관리법 규정상, 설치검사를 받지 않고 운행할 수 있습니까?",
         "승강기의 제조·수입업자 또는 관리주체는 설치검사를 받지 아니하거나 설치검사에 불합격한 승강기를 운행하게 하거나 운행하여서는 아니 됩니다. 이는 승강기 안전관리법 제28조 제2항에 명시되어 있습니다."),

        ("안전", "가로등이 꺼져서 밤에 위험합니다.",
         "가로등 고장 신고 감사합니다. 시설관리과에서 현장 확인 후 수리 조치하겠습니다. 정확한 위치(가로등 번호, 도로명 등)를 알려주시면 신속한 처리가 가능합니다. 야간 안전을 위해 2일 이내 수리를 완료하도록 하겠습니다."),

        ("안전", "재난 대피소 위치를 알고 싶습니다.",
         "재난 대피소는 국민재난안전포털(www.safekorea.go.kr)에서 검색하실 수 있습니다. 주소 또는 현재 위치 기반으로 가까운 대피소를 확인할 수 있으며, 안전디딤돌 앱에서도 조회 가능합니다. 대피소에는 생활필수품과 의료용품이 비치되어 있습니다."),

        ("안전", "재난안전상황실은 어떤 요건을 갖추어야 하나요?",
         "재난 및 안전관리 기본법 시행령 제23조 제1항에 따르면, 재난안전상황실은 신속한 재난정보의 수집과 전파, 재난대비 자원의 관리 및 지원을 위한 재난방송과 정보통신체계, 재난상황의 효율적 관리를 위한 장비 운영 및 관리체계, 전담인력과 운영규정을 갖추어야 합니다."),

        ("안전", "재난이 발생하거나 발생할 우려가 있는 경우, 시장이나 군수, 구청장이 할 수 있는 조치는 무엇인가요?",
         "시장, 군수, 구청장과 지역통제단장은 사람의 생명 또는 신체나 재산에 대한 위해를 방지하기 위하여 해당 지역 주민이나 그 지역 안에 있는 사람에게 대피하도록 명령할 수 있습니다. 또한 선박, 자동차 등을 대피시킬 것을 명할 수 있으며, 미리 대피장소를 지정할 수 있습니다."),

        ("안전", "소방청장은 소방사업자에게 어떤 경우 신고를 요구할 수 있나요?",
         "소방산업의 진흥에 관한 법률 제17조 제1항에 따르면 소방청장은 소방장비 및 인력의 이용촉진 등 소방산업의 진흥을 위하여 필요한 경우 소방사업자로 하여금 그 사업의 범위와 내용 등을 신고하게 할 수 있으며, 신고사항이 변경되는 경우에도 신고하게 할 수 있습니다."),

        ("안전", "화재 발생 시 소화기 사용법을 알려주세요.",
         "소화기 사용법은 다음과 같습니다. 1) 안전핀을 뽑습니다. 2) 노즐을 불이 난 곳을 향합니다. 3) 손잡이를 힘껏 움켜쥡니다. 4) 빗자루로 쓸듯이 골고루 뿌립니다. 소화기는 바람을 등지고 사용하시고, 사용 거리는 3~5m가 적당합니다."),
    ]

    samples = []
    for cat, question, answer in examples:
        text = (
            f"[|system|]{system_msg}[|endofturn|]\n"
            f"[|user|]다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n"
            f"[카테고리: {cat}]\n"
            f"민원 내용: {question}[|endofturn|]\n"
            f"[|assistant|]{answer}[|endofturn|]"
        )
        samples.append(text)

    # 56개 예제를 n_samples까지 반복 확장 (각 반복마다 셔플하여 순서 다양화)
    if len(samples) < n_samples:
        base_samples = list(samples)
        while len(samples) < n_samples:
            batch = list(base_samples)
            random.shuffle(batch)
            samples.extend(batch)
        samples = samples[:n_samples]

    random.shuffle(samples)

    # 카테고리 분포 확인
    cat_counts = {}
    for s in samples:
        for cat in ["교통", "행정", "환경", "세금", "기타", "건축", "복지", "안전"]:
            if f"[카테고리: {cat}]" in s:
                cat_counts[cat] = cat_counts.get(cat, 0) + 1
                break

    print(f"  인라인 캘리브레이션 샘플 {len(samples)}개 생성 완료")
    print(f"  카테고리 분포: {cat_counts}")
    return samples

calib_data = prepare_calibration_data(n_samples=CALIB_N_SAMPLES, seed=CALIB_SEED)
wandb.log({"calibration_samples": len(calib_data)})

# ── Step 2: AutoAWQ로 모델 로드 ──
print("\n[2/4] AutoAWQ로 머지 모델 로드...")

# AWQ 로드 전 GPU 메모리 확보
gc.collect()
torch.cuda.empty_cache()

from awq import AutoAWQForCausalLM

# AutoAWQ의 from_pretrained는 CPU에 로드 후 양자화 시 GPU로 이동
# L4(24GB)에서 안전하게 동작하도록 device_map 설정 없이 로드
# (AutoAWQ가 내부적으로 메모리 관리)
model = AutoAWQForCausalLM.from_pretrained(
    MERGED_OUTPUT_DIR,
    safetensors=True,
    trust_remote_code=True,
    device_map=None,  # AutoAWQ 내부에서 관리하도록 위임
)
print("  모델 로드 완료")

mem_before_quant = torch.cuda.memory_allocated() / 1024**3
print(f"  GPU 메모리 (양자화 전): {mem_before_quant:.2f} GB")

# ── Step 3: AWQ 양자화 수행 ──
print("\n[3/4] AWQ 양자화 수행...")
print(f"  양자화 설정: {AWQ_QUANT_CONFIG}")
print(f"  캘리브레이션 샘플: {len(calib_data)}개")
print(f"  예상 소요 시간: 20~40분 (GPU에 따라 다름)")

quant_actual_start = time.time()
model.quantize(
    tokenizer,
    quant_config=AWQ_QUANT_CONFIG,
    calib_data=calib_data,
)
quant_elapsed = time.time() - quant_actual_start
print(f"  양자화 완료! 소요 시간: {quant_elapsed:.1f}초 ({quant_elapsed/60:.1f}분)")

# ── Step 4: 양자화 모델 저장 ──
print(f"\n[4/4] 양자화 모델 저장: {AWQ_OUTPUT_DIR}")
os.makedirs(AWQ_OUTPUT_DIR, exist_ok=True)
model.save_quantized(AWQ_OUTPUT_DIR, safetensors=True)
tokenizer.save_pretrained(AWQ_OUTPUT_DIR)

# modeling_exaone.py, configuration_exaone.py 복사 (커스텀 코드)
import shutil
for fname in ["modeling_exaone.py", "configuration_exaone.py"]:
    src = os.path.join(MERGED_OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(AWQ_OUTPUT_DIR, fname))
        print(f"  {fname} 복사 완료")

# 모델 크기 비교
awq_size_bytes = sum(
    os.path.getsize(os.path.join(AWQ_OUTPUT_DIR, f))
    for f in os.listdir(AWQ_OUTPUT_DIR)
    if f.endswith(('.safetensors', '.bin'))
)
awq_size_gb = awq_size_bytes / 1024**3
compression_ratio = merged_size_gb / awq_size_gb if awq_size_gb > 0 else 0
size_reduction_pct = (1 - awq_size_gb / merged_size_gb) * 100 if merged_size_gb > 0 else 0

print(f"\n  크기 비교:")
print(f"    머지 모델 (BF16): {merged_size_gb:.2f} GB")
print(f"    AWQ 모델 (W4):    {awq_size_gb:.2f} GB")
print(f"    압축률:           {compression_ratio:.2f}x")
print(f"    크기 감소:        {size_reduction_pct:.1f}%")

wandb.log({
    "quantization_time_seconds": quant_elapsed,
    "awq_model_size_gb": awq_size_gb,
    "merged_model_size_gb": merged_size_gb,
    "compression_ratio": compression_ratio,
    "size_reduction_pct": size_reduction_pct,
})

# 양자화 로그 저장
quant_log = {
    "stage": "2_awq_quantization_v2",
    "timestamp": datetime.now().isoformat(),
    "merged_model_dir": MERGED_OUTPUT_DIR,
    "output_dir": AWQ_OUTPUT_DIR,
    "quant_config": AWQ_QUANT_CONFIG,
    "calibration_samples": len(calib_data),
    "awq_model_size_gb": awq_size_gb,
    "merged_model_size_gb": merged_size_gb,
    "compression_ratio": compression_ratio,
    "size_reduction_pct": size_reduction_pct,
    "quantization_time_seconds": quant_elapsed,
}
with open(os.path.join(AWQ_OUTPUT_DIR, "quantization_log.json"), "w") as f:
    json.dump(quant_log, f, indent=2, ensure_ascii=False)

total_stage2_time = time.time() - quant_start_time
print(f"\n{'=' * 60}")
print(f"Stage 2 완료! 소요 시간: {total_stage2_time:.1f}초 ({total_stage2_time/60:.1f}분)")
print(f"AWQ 모델 저장 위치: {AWQ_OUTPUT_DIR}")
print(f"{'=' * 60}")

# 메모리 정리
del model
gc.collect()
torch.cuda.empty_cache()
print("\nGPU 메모리 해제 완료")

Stage 2: AWQ 양자화 (W4A16g128)
  GPU 메모리 정리 후: 2.14 GB 사용 중

[1/4] 캘리브레이션 데이터 준비...
  GitHub에서 다운로드 시도: https://raw.githubusercontent.com/GovOn-org/GovOn/develop/data/processed/v2_trai...
  GitHub 다운로드 데이터 사용: /tmp/v2_train_calib.jsonl
  캘리브레이션 샘플 512개 로드 완료

[2/4] AutoAWQ로 머지 모델 로드...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  모델 로드 완료
  GPU 메모리 (양자화 전): 2.14 GB

[3/4] AWQ 양자화 수행...
  양자화 설정: {'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}
  캘리브레이션 샘플: 512개
  예상 소요 시간: 20~40분 (GPU에 따라 다름)


AWQ: 100%|██████████| 32/32 [18:35<00:00, 34.85s/it]


  양자화 완료! 소요 시간: 1117.2초 (18.6분)

[4/4] 양자화 모델 저장: /content/models/awq_quantized_model
  modeling_exaone.py 복사 완료
  configuration_exaone.py 복사 완료

  크기 비교:
    머지 모델 (BF16): 14.56 GB
    AWQ 모델 (W4):    4.94 GB
    압축률:           2.95x
    크기 감소:        66.1%

Stage 2 완료! 소요 시간: 1139.8초 (19.0분)
AWQ 모델 저장 위치: /content/models/awq_quantized_model

GPU 메모리 해제 완료


---
## 6. AWQ 모델 추론 검증

양자화된 모델이 정상적으로 동작하는지 추론 테스트를 수행한다.

In [8]:
# ============================================================
# AWQ 모델 추론 검증
# ============================================================
print("=" * 60)
print("AWQ 모델 추론 검증")
print("=" * 60)

from transformers import AutoTokenizer, AutoModelForCausalLM

print("\n[1/2] AWQ 모델 로드...")
awq_tokenizer = AutoTokenizer.from_pretrained(AWQ_OUTPUT_DIR, trust_remote_code=True)
awq_model = AutoModelForCausalLM.from_pretrained(
    AWQ_OUTPUT_DIR,
    device_map="auto",
    trust_remote_code=True,
)
awq_model.eval()

mem_awq = torch.cuda.memory_allocated() / 1024**3
print(f"  GPU 메모리 사용: {mem_awq:.2f} GB")

print("\n[2/2] 추론 테스트...")

eos_ids = get_eos_ids(awq_tokenizer)

test_prompts_awq = [
    "주민등록증 재발급 절차가 어떻게 되나요?",
    "우리 동네 도로에 포트홀이 생겨서 위험합니다.",
    "기초생활수급자 신청 방법을 알려주세요.",
]

awq_results = []
for i, prompt in enumerate(test_prompts_awq):
    messages = [
        {"role": "system", "content": "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."},
        {"role": "user", "content": f"다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n민원 내용: {prompt}"},
    ]
    input_ids = awq_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(awq_model.device)

    with torch.no_grad():
        output = awq_model.generate(
            input_ids,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos_ids,
        )

    response = awq_tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=False)
    clean_response = re.sub(r"<thought>.*?</thought>", "", response, flags=re.DOTALL).strip()
    has_eos = any(tok in response for tok in ['[|endofturn|]', '<|endofturn|>'])

    print(f"\n  --- 테스트 {i+1} ---")
    print(f"  질의: {prompt}")
    print(f"  응답 (300자): {clean_response[:300]}")
    print(f"  EOS 생성: {'Yes' if has_eos else 'No'}")

    awq_results.append({
        "prompt": prompt,
        "response": clean_response[:500],
        "has_eos": has_eos,
    })

wandb.log({
    "awq_gpu_mem_gb": mem_awq,
    "awq_inference_results": awq_results,
})

print(f"\n{'=' * 60}")
print("AWQ 모델 추론 검증 완료")
print(f"{'=' * 60}")

# 메모리 정리
del awq_model
gc.collect()
torch.cuda.empty_cache()

AWQ 모델 추론 검증

[1/2] AWQ 모델 로드...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  GPU 메모리 사용: 7.08 GB

[2/2] 추론 테스트...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  --- 테스트 1 ---
  질의: 주민등록증 재발급 절차가 어떻게 되나요?
  응답 (300자): 주민등록증 재발급 신청 방법은 다음과 같습니다. 먼저, 주민등록증 재발급 신청서를 작성하시면 됩니다. 이 신청서에는 사진 1장과 신분증 사본이 필요합니다. 또한, 재발급 사유와 발급일자 등을 기재해야 합니다. 다음으로, 주민센터나 온라인으로 신청하시게 됩니다. 그리고 재발급 신청 후, 주민센터에서 발급받으실 수 있으며, 재발급 수수료는 5000원입니다. 재발급 수수료는 현금이나 신용카드로 납부 가능합니다. 마지막으로, 재발급된 주민등록증은 2년 이내에만 유효하며, 그 이후에는 다시 재발급을 받으셔야 합니다.[|endofturn|]
  EOS 생성: Yes


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  --- 테스트 2 ---
  질의: 우리 동네 도로에 포트홀이 생겨서 위험합니다.
  응답 (300자): 안녕하십니까?
귀하께서 응답소를 통해 신청하신 민원에 대한 검토 결과를 다음과 같이 알려드립니다.
먼저, 귀하의 민원내용은 "도로 포트홀 보수 요청"으로 판단됩니다.
귀하의 민원사항에 대하여 아래와 같이 답변드리고자 합니다.
우선, 해당 구간(도로)에 대해서는 포트홀보수를 완료하였으며, 추가로 발생된 포트홀로 인하여 불편함을 겪으셨다면, 해당구간을 관할하는 도로관리부서에 신고하여 조치할 수 있도록 하겠습니다.
아울러, 포트홀 발생 시 신고해 주시면 빠른 시일 내에 보수가 이루어지고 있으니, 앞으로도 지속적인 관심 부탁드립니다.
다시
  EOS 생성: Yes

  --- 테스트 3 ---
  질의: 기초생활수급자 신청 방법을 알려주세요.
  응답 (300자): 안녕하십니까?
귀하께서 응답소를 통해 신청하신 민원에 대한 검토 결과를 다음과 같이 알려드립니다.
귀하의 민원내용은 "기초생활수급자 신청"에 관한 것으로 판단됩니다.
귀하의 질의사항에 대해 검토한 의견은 다음과 같습니다.
가. 기초생활수급자는 국민기초생활보장법 제1조제2호 및 동법 시행령 제3조에 의거, 국민기초생활보장법상 수급권자로 인정되어 기초생활보장사업에 참여하여 지원받으실 수 있으며,
나. 기초생활수급자는 국민기초생활보장법 제5조에 따라, 가구당 소득 및 재산 기준에 해당할 경우, 기초생활보장사업에 참여하여 지원받으실 수 있
  EOS 생성: No

AWQ 모델 추론 검증 완료


---
## 7. HuggingFace Hub 업로드

머지 모델과 AWQ 모델을 HuggingFace Hub에 업로드한다.

In [9]:
# ============================================================
# HuggingFace Hub 업로드
# ============================================================
from huggingface_hub import HfApi, login

print("=" * 60)
print("HuggingFace Hub 업로드")
print("=" * 60)

# HuggingFace 로그인
login(token=os.environ["HF_TOKEN"])
api = HfApi()

# ── 머지 모델 업로드 ──
print(f"\n[1/2] 머지 모델 업로드: {HF_MERGED_REPO}")
print(f"  소스: {MERGED_OUTPUT_DIR}")
print(f"  예상 크기: {merged_size_gb:.2f} GB (시간이 다소 걸릴 수 있음)")

try:
    api.create_repo(HF_MERGED_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=MERGED_OUTPUT_DIR,
        repo_id=HF_MERGED_REPO,
        commit_message="Upload GovOn-EXAONE-Merged-v2 (BF16)",
    )
    print(f"  [OK] 머지 모델 업로드 완료: https://huggingface.co/{HF_MERGED_REPO}")
    wandb.log({"hf_merged_repo": HF_MERGED_REPO, "hf_merged_upload": True})
except Exception as e:
    print(f"  [ERROR] 머지 모델 업로드 실패: {e}")
    wandb.log({"hf_merged_upload": False, "hf_merged_error": str(e)})

# ── AWQ 모델 업로드 ──
print(f"\n[2/2] AWQ 모델 업로드: {HF_AWQ_REPO}")
print(f"  소스: {AWQ_OUTPUT_DIR}")
print(f"  예상 크기: {awq_size_gb:.2f} GB")

try:
    api.create_repo(HF_AWQ_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=AWQ_OUTPUT_DIR,
        repo_id=HF_AWQ_REPO,
        commit_message="Upload GovOn-EXAONE-AWQ-v2 (W4A16g128)",
    )
    print(f"  [OK] AWQ 모델 업로드 완료: https://huggingface.co/{HF_AWQ_REPO}")
    wandb.log({"hf_awq_repo": HF_AWQ_REPO, "hf_awq_upload": True})
except Exception as e:
    print(f"  [ERROR] AWQ 모델 업로드 실패: {e}")
    wandb.log({"hf_awq_upload": False, "hf_awq_error": str(e)})

print(f"\n{'=' * 60}")
print("HuggingFace Hub 업로드 완료")
print(f"{'=' * 60}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace Hub 업로드

[1/2] 머지 모델 업로드: umyunsang/GovOn-EXAONE-Merged-v2
  소스: /content/models/merged_model
  예상 크기: 14.56 GB (시간이 다소 걸릴 수 있음)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0003-of-00004.safetensors:   0%|          | 22.6MB / 4.92GB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00002.safetensors:   0%|          | 61.5kB / 4.47GB            

  ...0002-of-00002.safetensors:   4%|4         | 35.6MB /  839MB            

  [OK] AWQ 모델 업로드 완료: https://huggingface.co/umyunsang/GovOn-EXAONE-AWQ-v2

HuggingFace Hub 업로드 완료


---
## 8. 최종 요약 및 WandB 완료

In [10]:
# ============================================================
# 최종 요약 및 WandB 완료
# ============================================================
total_elapsed = time.time() - merge_start_time

print("=" * 60)
print("파이프라인 완료 요약")
print("=" * 60)
print(f"")
print(f"  베이스 모델:       {BASE_MODEL_ID}")
print(f"  LoRA 어댑터:       {ADAPTER_ID}")
print(f"  EXAONE 리비전:     {EXAONE_REVISION}")
print(f"")
print(f"  머지 모델 크기:    {merged_size_gb:.2f} GB (BF16)")
print(f"  AWQ 모델 크기:     {awq_size_gb:.2f} GB (W4A16g128)")
print(f"  압축률:            {compression_ratio:.2f}x")
print(f"  크기 감소:         {size_reduction_pct:.1f}%")
print(f"")
print(f"  머지 소요 시간:    {merge_elapsed:.1f}초 ({merge_elapsed/60:.1f}분)")
print(f"  양자화 소요 시간:  {quant_elapsed:.1f}초 ({quant_elapsed/60:.1f}분)")
print(f"  전체 소요 시간:    {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")
print(f"")
print(f"  HF 머지 모델:      https://huggingface.co/{HF_MERGED_REPO}")
print(f"  HF AWQ 모델:       https://huggingface.co/{HF_AWQ_REPO}")
print(f"")
print(f"{'=' * 60}")

# WandB 최종 로깅
wandb.log({
    "total_pipeline_time_seconds": total_elapsed,
    "total_pipeline_time_minutes": total_elapsed / 60,
})

# WandB summary에 주요 지표 기록
wandb.summary["merged_model_size_gb"] = merged_size_gb
wandb.summary["awq_model_size_gb"] = awq_size_gb
wandb.summary["compression_ratio"] = compression_ratio
wandb.summary["size_reduction_pct"] = size_reduction_pct
wandb.summary["total_time_minutes"] = total_elapsed / 60
wandb.summary["hf_merged_repo"] = HF_MERGED_REPO
wandb.summary["hf_awq_repo"] = HF_AWQ_REPO

wandb.finish()
print("\nWandB 로깅 완료")

파이프라인 완료 요약

  베이스 모델:       LGAI-EXAONE/EXAONE-Deep-7.8B
  LoRA 어댑터:       umyunsang/GovOn-EXAONE-LoRA-v2
  EXAONE 리비전:     17b70148e344

  머지 모델 크기:    14.56 GB (BF16)
  AWQ 모델 크기:     4.94 GB (W4A16g128)
  압축률:            2.95x
  크기 감소:         66.1%

  머지 소요 시간:    138.8초 (2.3분)
  양자화 소요 시간:  1117.2초 (18.6분)
  전체 소요 시간:    2205.4초 (36.8분)

  HF 머지 모델:      https://huggingface.co/umyunsang/GovOn-EXAONE-Merged-v2
  HF AWQ 모델:       https://huggingface.co/umyunsang/GovOn-EXAONE-AWQ-v2



adapter_trainable_params,▁
adapter_trainable_pct,▁
awq_gpu_mem_gb,▁
awq_model_size_gb,▁
base_model_params,▁
calibration_samples,▁▁
compression_ratio,▁
gpu_mem_after_merge_gb,▁
gpu_mem_base_model_gb,▁
merge_time_seconds,▁
+6,...



WandB 로깅 완료
